<a href="https://colab.research.google.com/github/MahinourAbdelgawad/Plant-Disease-Detection/blob/main/plant_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Plant Species and Disease Detection
Possible scopes:
- Predict species
- Predict diseased or healthy
- Predict diseased or healthy given species
- Predict disease
- Predict disease given species

Methods?
- Classical CV: Can predict diseased vs. healthy by detecting color shifts (green vs brown spots) and leaf shapes, but completely blind to the small differences between specific diseases or plant species
- Classical ML: Can predict diseased vs. healthy, predict species, and predict disease given species if a human manually calculates and inputs the mathematical features (like texture math or spot sizes)
- Deep Learning: Can do all scopes

Possible paths:
- Predict diseased or healthy using classical CV and then classical ML and then DL and compare

In [17]:
# FOR RUNNING LOCALLY
! pip install matplotlib opencv-python numpy seaborn pandas tqdm scikit-learn kagglehub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 501.6 kB/s eta 0:00:000:00:01m eta 0:00:01
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 1.5 MB/s eta 0:00:00m eta 0:00:010:00:010m
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 1.7 MB/s eta 0:00:00 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 3.3 MB/s eta 0:00:00m eta 0:00:010:00:01


### STEP 0: Load dataset, examine data, and preprocess

In [6]:
import matplotlib.pyplot as plt
import cv2 as cv
import pandas as pd
import seaborn as sns
import numpy as np

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

In [7]:
#LOCAL VERSION
! pip install python-dotenv
import os
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get('KAGGLE_API_TOKEN'):
    print("Error: Could not find 'KAGGLE_API_TOKEN'")

In [9]:
! pip install kagglehub
import kagglehub

path = kagglehub.dataset_download("riteshmaurya86/cleaned-plant-disease-image-dataset")

print("Dataset downloaded to:", path)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 465.8 kB/s eta 0:00:001m992.2 kB/s eta 0:00:01
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 620.5 kB/s eta 0:00:00MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.5 MB/s eta 0:00:00
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.6.17-py3-none-any.whl.metadata (2.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.8/243.8 kB 1.8 MB/s eta 0:00:002.4 MB/s eta 0:00:01
Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (807 kB)
Using cached requests-2.34.2-py3-none-a

/media/mahi/Shared/AUC/Semesters/6.5- Summer 2026/Fundamentals of Computer Vision/Project/Plant-Disease-Detection/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 1.34G/1.34G [21:26<00:00, 1.12MB/s]

Extracting files...


Dataset downloaded to: /home/mahi/.cache/kagglehub/datasets/riteshmaurya86/cleaned-plant-disease-image-dataset/versions/1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/PlantDetection'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [10]:
# LOCAL VERSION
import os
OUTPUT_DIR = os.path.expanduser('outputs') 
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [11]:
# check out the dataset
import os

DATA_DIR = os.path.join(path, 'data')
TRAIN_DIR = os.path.join(DATA_DIR, 'train')

classes = sorted(os.listdir(TRAIN_DIR))
print(f"Number of classes: {len(classes)}")
for c in classes:
    n = len(os.listdir(os.path.join(TRAIN_DIR, c)))
    print(f"{c}: {n} images")

Number of classes: 38
Apple___Apple_scab: 2016 images
Apple___Black_rot: 1987 images
Apple___Cedar_apple_rust: 1760 images
Apple___healthy: 2008 images
Blueberry___healthy: 1816 images
Cherry_(including_sour)___Powdery_mildew: 1683 images
Cherry_(including_sour)___healthy: 1826 images
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 1642 images
Corn_(maize)___Common_rust_: 1907 images
Corn_(maize)___Northern_Leaf_Blight: 1908 images
Corn_(maize)___healthy: 1859 images
Grape___Black_rot: 1888 images
Grape___Esca_(Black_Measles): 1920 images
Grape___Leaf_blight_(Isariopsis_Leaf_Spot): 1722 images
Grape___healthy: 1692 images
Orange___Haunglongbing_(Citrus_greening): 2010 images
Peach___Bacterial_spot: 1838 images
Peach___healthy: 1728 images
Pepper,_bell___Bacterial_spot: 1913 images
Pepper,_bell___healthy: 1988 images
Potato___Early_blight: 1939 images
Potato___Late_blight: 1939 images
Potato___healthy: 1824 images
Raspberry___healthy: 1781 images
Soybean___healthy: 2022 images
Squas

In [ ]:
# visualize some samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
sample_classes = classes[:8]

for ax, c in zip(axes.flat, sample_classes):
    img_name = os.listdir(os.path.join(TRAIN_DIR, c))[0]
    img = cv.imread(os.path.join(TRAIN_DIR, c, img_name))
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(c, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# dataset summary
import os
data_list = []
for c in classes:
    num_images = len(os.listdir(os.path.join(TRAIN_DIR, c)))


    if "___" in c:
        plant, disease = c.split("___")
    else:
        plant, disease = c, "Unknown"

    data_list.append({
        'Folder_Name': c,
        'Plant': plant.replace('_', ' '),
        'Condition': disease.replace('_', ' '),
        'Image_Count': num_images
    })

df = pd.DataFrame(data_list)

total_images = df['Image_Count'].sum()
unique_plants = df['Plant'].nunique()
unique_conditions = df[df['Condition'] != 'healthy']['Condition'].nunique()

print("="*40)
print(f"DATASET SUMMARY")
print("="*40)
print(f"Total Images: {total_images:,}")
print(f"Unique Plants: {unique_plants} {df['Plant'].unique().tolist()}")
print(f"Unique Diseases: {unique_conditions}")
print(f"Average images/class: {df['Image_Count'].mean():.1f}")
print("="*40)

In [ ]:
# experimemt with preprocessing
class_index = 0
image_index = 777

canny_threshold1 = 40
canny_threshold2 = 100

img = cv.imread(os.path.join(TRAIN_DIR, classes[class_index], os.listdir(os.path.join(TRAIN_DIR, classes[class_index]))[image_index]))

img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

# Grayscale
gray = cv.cvtColor(img, cv.COLOR_RGB2GRAY)

# Gaussian filter
blurred = cv.GaussianBlur(gray, (5, 5), 0)

# Canny edge detectio
edges = cv.Canny(blurred, threshold1=canny_threshold1, threshold2=canny_threshold2)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Original', 'Grayscale', 'Filtered (Gaussian Blur)', 'Edges (Canny)']
imgs = [img, gray, blurred, edges]
cmaps = [None, 'gray', 'gray', 'gray']
for ax, im, t, cm in zip(axes, imgs, titles, cmaps):
    ax.imshow(im, cmap=cm)
    ax.set_title(t)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
def preprocess(path):
  #TODO: come back to this, what preprocessing will we really need????
  img = cv.imread(path)
  img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
  # img = cv.resize(img, size)

  canny_threshold1 = 40
  canny_threshold2 = 100

  gray = cv.cvtColor(img, cv.COLOR_RGB2GRAY)
  blurred = cv.GaussianBlur(gray, (5, 5), 0)
  edges = cv.Canny(blurred, canny_threshold1, canny_threshold2)

  return img, gray, blurred, edges

## GOAL 1: DISEASED VS HEALTHY

In [12]:
# HELPER FUNCTION FOR ALL 3 APPROACHES
def get_label(class_name): # retrieve label for each class for training (healthy or diseases)
    return "healthy" if "healthy" in class_name.lower() else "diseased"

### Step 1: Classical CV only
Predict if a leaf is diseased or healthy based on a rule-based heuristic and a threshold on a visual measurement (color).
Built on the ratio of green color in a lead (higher brown/yellow patches = diseased)
THis is based only on color and does not consider curving, etc..

In [14]:
def green_ratio(img):
    hsv = cv.cvtColor(img, cv.COLOR_RGB2HSV) #convert to hsv for easier color segmentation (focus on green hue)
    # green ranges: OpenCV (0–179 scale): Hue: 40–70, Saturation: 50–255, Value: 50–255
    #TODO: experiment with ranges
    lower_green = (25, 40, 40)
    upper_green = (90, 255, 255)

    mask = cv.inRange(hsv, lower_green, upper_green) #create mask of pixels in the green range (255 = in range, 0 = not)
    return mask.sum() / (mask.shape[0] * mask.shape[1] * 255) #find ratio and normalize between 0 and 1


# test function on a sample + select threshold
sample_ratios = {'healthy': [], 'diseased': []}
for c in classes:
    label = get_label(c)
    files = os.listdir(os.path.join(TRAIN_DIR, c))[:100]  # 30 samples per class for  now

    for f in files:
        img = cv.imread(os.path.join(TRAIN_DIR, c, f))
        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        sample_ratios[label].append(green_ratio(img))


print("Healthy median green ratio:", np.median(sample_ratios['healthy']))
print("Diseased median green ratio:", np.median(sample_ratios['diseased']))

# MEDIANS FOR 30 SAMPLES PER CLASS:
# Healthy median green ratio: 0.4345855712890625
# Diseased median green ratio: 0.39788055419921875
# FOR 60:
# Healthy median green ratio: 0.4390716552734375
# Diseased median green ratio: 0.40045928955078125
# FOR 100:
# Healthy median green ratio: 0.440948486328125
# Diseased median green ratio: 0.40067291259765625
# *SELECTED THRESHOLD: 0.42*

Healthy median green ratio: 0.44738006591796875
Diseased median green ratio: 0.40235137939453125


In [15]:
def classical_cv_predict(img, threshold = 0.42):
    return "healthy" if green_ratio(img) >= threshold else "diseased"

In [16]:
# RUN! 
# run classical cv predictor on full dataset and save results
import pandas as pd
from tqdm import tqdm

threshold = 0.42 # threshold selected based on medians above
results = []
for c in tqdm(classes):
    label = get_label(c)
    class_dir = os.path.join(TRAIN_DIR, c)

    for f in os.listdir(class_dir):
        img = cv.imread(os.path.join(class_dir, f))
        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

        ratio = green_ratio(img)
        pred = "healthy" if ratio >= threshold else "diseased"
        results.append({
            'class': c,
            'file': f,
            'true_label': label,
            'green_ratio': ratio,
            'pred_label': pred,
            'correct': pred == label
        })

df_cv = pd.DataFrame(results)
df_cv.to_csv(os.path.join(OUTPUT_DIR, 'classical_cv_predictions.csv'), index=False)

100%|██████████| 38/38 [00:46<00:00,  1.23s/it]


In [18]:
# Compute metrics and save for report purposes
# metrics: classification report, confusion matrix
from sklearn.metrics import classification_report, confusion_matrix
import json

report = classification_report(df_cv['true_label'], df_cv['pred_label'], output_dict=True)
cm = confusion_matrix(df_cv['true_label'], df_cv['pred_label'], labels=['healthy', 'diseased'])

with open(os.path.join(OUTPUT_DIR, 'classical_cv_summary.json'), 'w') as f:
    json.dump({'report': report, 'confusion_matrix': cm.tolist()}, f, indent=2)

print(classification_report(df_cv['true_label'], df_cv['pred_label']))

              precision    recall  f1-score   support

    diseased       0.73      0.54      0.62     48001
     healthy       0.37      0.58      0.45     22294

    accuracy                           0.55     70295
   macro avg       0.55      0.56      0.54     70295
weighted avg       0.62      0.55      0.57     70295



### Step 2: Classical ML

#### First: Feature extraction


In [ ]:
# color histogram: to identify diseased spots that have distinct colors (e.g. brown yellow black vs healthy green)
def color_histogram(img, bins=32): # bins is number of intervals/quantizatons
    hist = cv.calcHist([img], [0, 1, 2], None, [bins]*3, [0, 256]*3) # 32 x 32 x 32
    hist = cv.normalize(hist, hist).flatten() # scale and flatten to 1d array
    return hist

### Step 3: Deep Learning